## Overview {.unnumbered}

This tutorial introduces the **PyTorch building blocks** you will use throughout the course:

- Creating and manipulating tensors
- Encoding text as numbers
- Building a vocabulary from raw text
- Constructing one-hot vectors
- Running a forward pass through a simple MLP

By the end you should be able to turn a sentence into a numeric tensor and pass it through a neural network layer.

## Setup {.unnumbered}


In [ ]:
import torch
import torch.nn as nn
import numpy as np
import re
from collections import Counter


If you are in Google Colab, PyTorch is pre-installed.  
For a local environment, install with:

```bash
pip install torch
```

## Tensors — the Foundation {.unnumbered}

A **tensor** is a generalisation of a matrix to any number of dimensions.  
In PyTorch every piece of data — text, images, audio — is eventually a tensor.

### Creating Tensors {.unnumbered}


In [ ]:
# 1-D tensor (vector)
v = torch.tensor([1.0, 2.0, 3.0])
print(v)          # tensor([1., 2., 3.])
print(v.shape)    # torch.Size([3])

# 2-D tensor (matrix)
M = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
print(M)
print(M.shape)    # torch.Size([2, 2])

# Random tensors
x = torch.randn(3, 4)   # 3 rows, 4 columns — Gaussian noise
z = torch.zeros(2, 5)   # all zeros


### Basic Operations {.unnumbered}


In [ ]:
a = torch.tensor([1.0, 2.0, 3.0])
b = torch.tensor([4.0, 5.0, 6.0])

print(a + b)          # element-wise addition
print(a * b)          # element-wise multiplication
print(torch.dot(a, b))  # dot product → scalar

# Matrix multiplication
W = torch.randn(3, 4)
x = torch.randn(4)
out = W @ x           # shape: (3,)
print(out.shape)


### Tensor Data Types {.unnumbered}

| `dtype` | Use case |
|---------|----------|
| `torch.float32` | Default for model weights and inputs |
| `torch.long` (int64) | Class labels |
| `torch.bool` | Masks |


In [ ]:
labels = torch.tensor([0, 1, 1, 0], dtype=torch.long)
floats = torch.tensor([0.1, 0.9, 0.8, 0.2], dtype=torch.float32)


## Text Encoding {.unnumbered}

Computers cannot process raw text.  
We must convert each word (or character) into a number.

### Character-Level Encoding {.unnumbered}


In [ ]:
text = "hello"

# Encode each character as its Unicode code point
char_ids = [ord(c) for c in text]
print(char_ids)          # [104, 101, 108, 108, 111]

tensor = torch.tensor(char_ids)
print(tensor)


**Limitation**: Unicode integers have no semantic meaning — `h=104` and `H=72` look very different numerically but represent the same letter.

### Word-Level Encoding {.unnumbered}

We map each unique word to an integer index using a **vocabulary**.


In [ ]:
sentence = "the cat sat on the mat"
words = sentence.split()

# Build vocabulary (sorted for reproducibility)
vocab = {word: idx for idx, word in enumerate(sorted(set(words)))}
print(vocab)
# {'cat': 0, 'mat': 1, 'on': 2, 'sat': 3, 'the': 4}

# Encode the sentence
encoded = [vocab[w] for w in words]
print(encoded)           # [4, 0, 3, 2, 4, 1]

tensor = torch.tensor(encoded, dtype=torch.long)
print(tensor)


## Building a Vocabulary {.unnumbered}

For larger corpora we need a robust vocabulary builder that handles:
- unknown words (`<UNK>`)
- padding (`<PAD>`)
- minimum frequency thresholds


In [ ]:
class Vocabulary:
    PAD = "<PAD>"
    UNK = "<UNK>"

    def __init__(self, min_freq: int = 1):
        self.min_freq = min_freq
        self.word2idx: dict[str, int] = {self.PAD: 0, self.UNK: 1}
        self.idx2word: dict[int, str] = {0: self.PAD, 1: self.UNK}

    def build(self, sentences: list[str]) -> "Vocabulary":
        counts = Counter(
            word
            for sent in sentences
            for word in re.findall(r"\b[a-z]+\b", sent.lower())
        )
        for word, freq in sorted(counts.items()):
            if freq >= self.min_freq and word not in self.word2idx:
                idx = len(self.word2idx)
                self.word2idx[word] = idx
                self.idx2word[idx] = word
        return self

    def encode(self, sentence: str) -> list[int]:
        words = re.findall(r"\b[a-z]+\b", sentence.lower())
        return [self.word2idx.get(w, 1) for w in words]  # 1 = UNK

    def __len__(self) -> int:
        return len(self.word2idx)


# Example usage
corpus = [
    "earnings were strong and revenue grew",
    "revenue declined and costs were high",
    "strong earnings growth driven by revenue",
]

vocab = Vocabulary(min_freq=1).build(corpus)
print(f"Vocabulary size: {len(vocab)}")
print(vocab.encode("strong earnings were reported"))


## One-Hot Vectors {.unnumbered}

A **one-hot vector** is a binary vector with a single `1` at the position corresponding to the word index and `0` everywhere else.

$$
\mathbf{e}_i \in \{0,1\}^{|V|}, \quad e_{ij} = \begin{cases} 1 & j = i \\ 0 & \text{otherwise} \end{cases}
$$


In [ ]:
def one_hot(idx: int, vocab_size: int) -> torch.Tensor:
    vec = torch.zeros(vocab_size)
    vec[idx] = 1.0
    return vec


V = len(vocab)
word = "earnings"
idx = vocab.word2idx[word]

oh = one_hot(idx, V)
print(f"'{word}' → index {idx}, one-hot shape: {oh.shape}")
print(oh[:10])   # first 10 values


### Sentence as a Sum of One-Hot Vectors {.unnumbered}

A simple Bag-of-Words vector is the **sum** (or mean) of the one-hot vectors for each word:


In [ ]:
sentence = "strong earnings growth"
indices = vocab.encode(sentence)

bow_vector = torch.stack([one_hot(i, V) for i in indices]).sum(dim=0)
print(f"BoW vector shape : {bow_vector.shape}")
print(f"Non-zero entries : {bow_vector.nonzero().squeeze()}")


**Note**: In practice we use `sklearn.feature_extraction.text.TfidfVectorizer` for efficiency, but understanding the one-hot foundation clarifies what the vectorizer is doing.

## A Simple MLP Forward Pass {.unnumbered}

Now let's pass a text vector through a two-layer MLP.


In [ ]:
INPUT_DIM   = V       # vocabulary size
HIDDEN_DIM  = 16
NUM_CLASSES = 2       # positive / negative

mlp = nn.Sequential(
    nn.Linear(INPUT_DIM, HIDDEN_DIM),
    nn.ReLU(),
    nn.Linear(HIDDEN_DIM, NUM_CLASSES),
)

# Forward pass with a single BoW vector
x = bow_vector.unsqueeze(0)   # add batch dimension → shape (1, V)
logits = mlp(x)

print(f"Input shape  : {x.shape}")
print(f"Output shape : {logits.shape}")
print(f"Raw logits   : {logits}")


### Converting Logits to Probabilities {.unnumbered}


In [ ]:
probs = torch.softmax(logits, dim=1)
pred  = probs.argmax(dim=1).item()

labels = ["negative", "positive"]
print(f"Probabilities : {probs.detach().numpy().round(3)}")
print(f"Prediction    : {labels[pred]}")


:::{.callout-note}
The model is **untrained** — weights are random — so the prediction is meaningless.  
In Lab 1 you will train the model on real data. This tutorial only demonstrates the **mechanics** of the forward pass.
:::

## Connecting the Pieces {.unnumbered}

The complete pipeline from raw text to a neural network prediction:

```
sentence  →  tokenize  →  vocabulary lookup  →  one-hot / TF-IDF  →  MLP  →  class probabilities
```

In code:


In [ ]:
def classify(sentence: str, model: nn.Module, vocab: Vocabulary, V: int) -> str:
    model.eval()
    indices = vocab.encode(sentence)
    bow = torch.stack([one_hot(i, V) for i in indices]).sum(dim=0).unsqueeze(0)
    with torch.no_grad():
        logits = model(bow)
    pred = logits.argmax(dim=1).item()
    return ["negative", "positive"][pred]


result = classify("earnings were strong and revenue grew", mlp, vocab, V)
print(f"Predicted: {result}  (random weights — train first!)")


## Summary {.unnumbered}

| Concept | PyTorch Tool | Purpose |
|---------|-------------|---------|
| Tensor | `torch.tensor` | Numeric container for all data |
| Word index | `dict` / `Vocabulary` | Map words → integers |
| One-hot vector | `torch.zeros` + assignment | Sparse word representation |
| Linear layer | `nn.Linear` | Learnable weight matrix $W\mathbf{x} + b$ |
| Activation | `nn.ReLU` | Introduce non-linearity |
| Softmax | `torch.softmax` | Convert logits → probabilities |

In **Lab 1** you will:

1. Replace one-hot vectors with TF-IDF features (denser and more informative)
2. Train the MLP on labeled 10-K sentences
3. Evaluate accuracy on held-out data
